# TacticalVision AI — Plug-and-Play Kaggle GPU Worker (PnLCalib 3D + SOTA Teams)

Run this notebook on **Kaggle** with:
- **Accelerator:** `GPU T4 x2` or `GPU P100` (30 GB RAM)
- **Internet:** `ON`
- **Attached Dataset:** `tacticalvision-models` (containing the 8 core production weights; official PnLCalib `SV_kp` & `SV_lines` weights auto-download & cache automatically if not already in the dataset)

Run Cells **1 → 2 → 3** in order, verify `calibrator_backend: pnlcalib_3d` in `/health`, then copy the printed `https://xxxx.trycloudflare.com` URL into **TacticalVision AI → Analyze Match → Remote GPU (Kaggle / Colab)** and click **Test Connection**.

In [ ]:
# ==============================================================================
# CELL 1: Clone TacticalVision-AI + Official PnLCalib 3D & Download Weights
# ==============================================================================
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/Mazenalodini/TacticalVision-AI.git"
GIT_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/TacticalVision-AI")

PNLCALIB_REPO_URL = "https://github.com/mguti97/PnLCalib.git"
PNLCALIB_REPO_DIR = Path("/kaggle/working/PnLCalib")
PNLCALIB_WEIGHTS_DIR = Path("/kaggle/working/pnlcalib_weights")
PNLCALIB_WEIGHT_URLS = {
    "SV_kp": "https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_kp",
    "SV_lines": "https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_lines",
}

# 1. Clone or update TacticalVision-AI on GIT_BRANCH
if REPO_DIR.exists():
    print(f"Updating existing repo at {REPO_DIR} -> {GIT_BRANCH}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", GIT_BRANCH, "FETCH_HEAD"], check=True)
else:
    print(f"Cloning {REPO_URL} ({GIT_BRANCH})...")
    subprocess.run(["git", "clone", "-b", GIT_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

# 2. Clone official mguti97/PnLCalib for 3D HRNetV2-w48 + PnL camera calibration
if not (PNLCALIB_REPO_DIR / "model" / "cls_hrnet.py").exists():
    print(f"Cloning official PnLCalib 3D into {PNLCALIB_REPO_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", PNLCALIB_REPO_URL, str(PNLCALIB_REPO_DIR)], check=True)
else:
    print(f"✅ PnLCalib repo already present at {PNLCALIB_REPO_DIR}")

# 3. Install core GPU worker + PnLCalib dependencies
core_deps = [
    "ultralytics",
    "lap",
    "pyyaml",
    "scipy",
    "timm",
    "rapidocr-onnxruntime",
    "imageio-ffmpeg",
    "fastapi",
    "uvicorn",
    "python-multipart",
    "pydantic-settings",
    "httpx",
]
print("Installing GPU worker & PnLCalib dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + core_deps, check=True)

# Prefer onnxruntime-gpu on CUDA; fall back cleanly to onnxruntime CPU if wheel mismatch
ort_rc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "onnxruntime-gpu"],
    check=False,
).returncode
if ort_rc != 0:
    print("onnxruntime-gpu wheel unavailable — installing onnxruntime fallback...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "onnxruntime"], check=True)

# 4. Locate or download official PnLCalib SV_kp & SV_lines weights (cached in /kaggle/working/pnlcalib_weights)
PNLCALIB_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
kaggle_input = Path("/kaggle/input")
for wname, url in PNLCALIB_WEIGHT_URLS.items():
    existing_input = next((p for p in kaggle_input.rglob(wname) if p.is_file() and p.stat().st_size > 1_000_000), None) if kaggle_input.exists() else None
    dst = PNLCALIB_WEIGHTS_DIR / wname
    if existing_input is not None:
        print(f"✅ Found {wname} in Kaggle Dataset: {existing_input}")
        if not dst.exists():
            try:
                dst.symlink_to(existing_input)
            except OSError:
                pass
    elif dst.exists() and dst.stat().st_size > 1_000_000:
        print(f"✅ Cached {wname} already present ({dst.stat().st_size / (1024**2):.1f} MB)")
    else:
        print(f"Downloading official PnLCalib weight {wname} (~253 MB)...")
        tmp_dst = PNLCALIB_WEIGHTS_DIR / f"{wname}.part"
        urllib.request.urlretrieve(url, str(tmp_dst))
        tmp_dst.replace(dst)
        print(f"✅ Downloaded {wname} ({dst.stat().st_size / (1024**2):.1f} MB)")

os.environ["PNLCALIB_REPO_DIR"] = str(PNLCALIB_REPO_DIR)
os.environ["PNLCALIB_WEIGHTS_DIR"] = str(PNLCALIB_WEIGHTS_DIR)
print("✅ Cell 1 complete! Branch:", GIT_BRANCH, "| Working directory:", os.getcwd())


In [ ]:
# ==============================================================================
# CELL 2: Auto-Locate MODELS_DIR & Verify GPU + All 8 Models + PnLCalib 3D
# ==============================================================================
import os
import sys
from pathlib import Path
import torch

REPO_DIR = Path("/kaggle/working/TacticalVision-AI")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

EXPECTED_MODELS = [
    ("players_best.pt",),
    ("pitch_best.pt",),
    ("action_spotting_v1_int8.onnx", "action_spotting_v1.onnx"),
    ("action_spotting_v1.pt",),
    ("jersey_legibility_v1_int8.onnx", "jersey_legibility_v1.onnx"),
    ("jersey_legibility_v1.pt",),
    ("resnet152_tf2_backbone.onnx",),
    ("soccernet_pca512_tf2_weights.npz",),
]

def find_models_dir() -> Path:
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working/TacticalVision-AI/models")]
    for root in search_roots:
        if not root.exists():
            continue
        for hit in root.rglob("players_best.pt"):
            return hit.parent
    raise FileNotFoundError(
        "Could not locate 'players_best.pt' under /kaggle/input. "
        "Please attach your 'tacticalvision-models' Kaggle Dataset to this notebook."
    )

MODELS_DIR = find_models_dir()
os.environ["MODELS_DIR"] = str(MODELS_DIR)
os.environ.setdefault("PNLCALIB_REPO_DIR", "/kaggle/working/PnLCalib")
os.environ.setdefault("PNLCALIB_WEIGHTS_DIR", "/kaggle/working/pnlcalib_weights")

print(f"📁 MODELS_DIR = {MODELS_DIR} (read-only mount, zero file copying)")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / (1024 ** 3)
    print(f"⚡ CUDA GPU Detected: {torch.cuda.get_device_name(0)} — {vram:.1f} GB VRAM")
else:
    print("⚠️ WARNING: CUDA GPU not detected! Enable GPU T4/P100 in Kaggle Session Options.")

print("\n── 8-Model Production Artifact Checklist ──")
missing = []
for spec in EXPECTED_MODELS:
    found = next((MODELS_DIR / name for name in spec if (MODELS_DIR / name).exists()), None)
    if found is not None:
        size_mb = found.stat().st_size / (1024 ** 2)
        print(f"  ✅ {found.name:<36s} ({size_mb:6.1f} MB)")
    else:
        print(f"  ❌ {spec[0]:<36s} MISSING")
        missing.append(spec[0])

if missing:
    raise RuntimeError(f"Missing required model files in {MODELS_DIR}: {missing}")

from engine.pnlcalib_calibrator import check_pnlcalib_readiness
pnl_status = check_pnlcalib_readiness(models_dir=MODELS_DIR)
print("\n── 3D Pitch Calibration Readiness (PnLCalib) ──")
print(f"  Backend:        {pnl_status['calibrator_backend']}")
print(f"  PnLCalib Ready: {pnl_status['pnlcalib_ready']}")
print(f"  SV_kp:          {pnl_status['weights_kp']}")
print(f"  SV_lines:       {pnl_status['weights_lines']}")
print("\n✅ All 8 core models + PnLCalib 3D verified and ready!")


In [ ]:
# ==============================================================================
# CELL 3: Launch FastAPI GPU Worker (port 8001) + Cloudflare Quick Tunnel
# ==============================================================================
import os
import re
import stat
import subprocess
import sys
import threading
import time
import urllib.request
from pathlib import Path

REPO_DIR = Path("/kaggle/working/TacticalVision-AI")
os.chdir(REPO_DIR)

# 1. Download standalone cloudflared binary (zero account / token needed)
cf_bin = Path("/kaggle/working/cloudflared")
if not cf_bin.exists():
    print("Downloading standalone cloudflared binary...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(cf_bin),
    )
    cf_bin.chmod(cf_bin.stat().st_mode | stat.S_IEXEC)

# 2. Start uvicorn gpu_worker.server:app on port 8001
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["GPU_WORK_DIR"] = "/kaggle/working/tv_gpu_jobs"
env.setdefault("PNLCALIB_REPO_DIR", "/kaggle/working/PnLCalib")
env.setdefault("PNLCALIB_WEIGHTS_DIR", "/kaggle/working/pnlcalib_weights")

server_log_path = Path("/kaggle/working/gpu_worker_server.log")
server_log = open(server_log_path, "w", encoding="utf-8")
uvicorn_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "gpu_worker.server:app", "--host", "0.0.0.0", "--port", "8001"],
    cwd=str(REPO_DIR),
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=env,
)

# Wait for local /health to come online
ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2) as r:
            if r.status == 200:
                print("✅ Local GPU Worker Health:", r.read().decode("utf-8"), flush=True)
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    uvicorn_proc.terminate()
    raise RuntimeError("uvicorn failed to start! Check /kaggle/working/gpu_worker_server.log")

# Live log tailer: streams uvicorn & GPU job requests/progress directly to Kaggle stdout
stop_tail = threading.Event()

def _tail_server_log():
    with open(server_log_path, "r", encoding="utf-8", errors="replace") as rf:
        rf.seek(0, os.SEEK_END)
        while not stop_tail.is_set():
            line = rf.readline()
            if line:
                print(f"[GPU-Worker] {line.rstrip()}", flush=True)
            else:
                time.sleep(0.25)

tail_thread = threading.Thread(target=_tail_server_log, daemon=True)
tail_thread.start()

# 3. Launch Cloudflare Quick Tunnel and print the live URL prominently
tunnel_proc = subprocess.Popen(
    [str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8001", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

try:
    assert tunnel_proc.stdout is not None
    for line in tunnel_proc.stdout:
        m = url_pattern.search(line)
        if m and public_url is None:
            public_url = m.group(0)
            print("\n" + "=" * 72, flush=True)
            print("🚀 TACTICALVISION AI KAGGLE GPU WORKER IS LIVE (PnLCalib 3D)!", flush=True)
            print(f"🔗 Copy & Paste URL:  {public_url}", flush=True)
            print("=" * 72, flush=True)
            print("Keep this cell running while analyzing videos from your local studio.", flush=True)
            print("Live requests & progress logs will stream below...\n", flush=True)
except KeyboardInterrupt:
    print("Shutting down Kaggle GPU Worker and Cloudflare Tunnel...", flush=True)
finally:
    stop_tail.set()
    tunnel_proc.terminate()
    uvicorn_proc.terminate()
    server_log.close()
